# Biological hardware as a regularizer: CORnet-Z vs AlexNet across V1–V4

**Neuromatch Academy · Deep Learning · 3-week project (this notebook = the write-up + experiment)**

### One-sentence question
> Does a *brain-inspired* backbone (**CORnet-Z**) let a small trainable attention head learn a stimulus representation that aligns with the visual hierarchy — and **resists overfitting** small, noisy fMRI data — better than a standard CNN (**AlexNet**)?

### Hypothesis
Mimicking the *hardware* of the ventral stream (CORnet-Z's shallow, anatomically-named `V1→V2→V4→IT` stages) acts as a **natural regularizer for the software**. The sharpest form of the claim is about *scale*: the CORnet advantage should be **largest when data is scarce** (Kay, ~1.7k images, 1 subject) and **shrink as data grows** (NSD, ~9k images). That convergence is what distinguishes a *regularizer* from a merely better model.

### Design at a glance
| Piece | What we do | Measured by |
|---|---|---|
| **Baseline (no training)** | Frozen AlexNet & CORnet-Z → per-layer RDMs → align to each ROI's neural RDM | **RSA** (layer × ROI grid) |
| **Intervention (deep learning)** | Small **self-attention head** on the frozen backbone, **fine-tuned** with a differentiable RSA loss | RSA alignment |
| **Comparison / the claim** | Same head per backbone; held-out RSA, **train→held gap**, and a **data-size curve spanning two datasets** | held-out RSA vs # images |

> **RSA is the yardstick throughout** — we compare representational *geometry* (RDMs), never regress to voxels. That is what makes "hierarchy alignment" a well-defined quantity across datasets with different voxel counts.

### Two datasets, one adapter — matched V1–V4
Different fMRI datasets have different subjects, scanners, and voxel counts, so we **cannot** merge voxels. Instead each dataset loads behind a common `VisualDataset` interface exposing its *own* ordered, early→late ROIs, and the identical pipeline runs on each. Both datasets here expose **retinotopic V1–V4**, so the cross-dataset curve compares **like-for-like (V4→V4)**:

| Dataset | Images | ROIs (early → late) | Top ventral ROI |
|---|---|---|---|
| **Kay vim-1** (2008) | 1,750 | `V1, V2, V3, V4` | `V4` |
| **NSD** via **Algonauts 2023** | ~9,000 / subject | `V1, V2, V3, V4` (from `prf-visualrois`) | `V4` |

> ⚠️ **Why not just augment the images to get "more data"?** Don't. Each BOLD response is *locked to one specific stimulus*; flipping/cropping the image makes the recorded response a mislabeled target and corrupts the RSA objective. The valid way to add data is a second *real* dataset — hence NSD.


## How to run
- **Runtime → GPU** (Colab). CPU is fine for the RSA baseline, slow for training.
- Run top-to-bottom. Heavy cells cache their outputs.
- **Kay** downloads automatically. **NSD** is access-gated (free): register at
  `algonautsproject.com/2023`, agree to the NSD terms, download one subject (e.g. `subj01`), and point
  `ALGONAUTS_ROOT` (cell 1b) at the extracted folder. Until then the notebook runs Kay-only and the
  scaling section shows the within-Kay curve with a note.
- Self-checks are printed, not assumed: loaders print keys/shapes/ROI voxel counts on load.


## 0 · Setup & imports

In [ ]:
import importlib, subprocess, sys
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
try:
    import cornet  # noqa: F401
except ImportError:
    pip("git+https://github.com/dicarlolab/CORnet")
# torch/torchvision/scipy/PIL/sklearn are preinstalled on Colab.
print("setup ok")

In [ ]:
import os, urllib.request, warnings, glob
from dataclasses import dataclass
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from scipy.stats import spearmanr
from PIL import Image
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 2024
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch:", torch.__version__)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

## 1 · The dataset adapter + shared RSA helpers

`VisualDataset` is the single interface the whole pipeline consumes. Everything downstream is dataset-agnostic — add a third dataset later by writing one more `load_*` function that returns this object.


In [ ]:
@dataclass
class VisualDataset:
    name: str
    stim: np.ndarray            # (N,H,W) gray  OR  (N,H,W,3) color
    resp: np.ndarray            # (N, n_voxels) responses aligned to stim
    roi_masks: dict             # roi_name -> bool mask over voxels
    roi_order: list             # roi names ordered early -> late
    top_roi: str                # top ventral ROI (training / cross-dataset target)
    def __repr__(self):
        rc = {r: int(self.roi_masks[r].sum()) for r in self.roi_order}
        return f"VisualDataset({self.name}: N={len(self.stim)} stim, ROIs {rc}, top={self.top_roi})"

# ---- shared RSA machinery (numpy) ------------------------------------
def rdm_from_responses(resp):
    # correlation-distance RDM (1 - Pearson r) across stimuli; resp: (N, voxels)
    r = resp - resp.mean(axis=1, keepdims=True)
    r = r / (np.linalg.norm(r, axis=1, keepdims=True) + 1e-8)
    return 1.0 - (r @ r.T)

def upper(m):
    return m[np.triu_indices(m.shape[0], k=1)]

def rsa(a, b):
    return spearmanr(upper(a), upper(b)).correlation

def noise_ceiling(resp_roi, n_splits=25, max_stim=1200, rng=None):
    # split-half reliability of the ROI's RDM -> the ceiling any model can reach.
    # Subsample stimuli so the NxN RDM stays bounded on large datasets (e.g. NSD).
    rng = rng or np.random.default_rng(SEED)
    if resp_roi.shape[0] > max_stim:
        resp_roi = resp_roi[rng.choice(resp_roi.shape[0], max_stim, replace=False)]
    v = resp_roi.shape[1]; out = []
    for _ in range(n_splits):
        p = rng.permutation(v); h1, h2 = p[: v // 2], p[v // 2:]
        out.append(rsa(rdm_from_responses(resp_roi[:, h1]),
                       rdm_from_responses(resp_roi[:, h2])))
    return float(np.mean(out)), float(np.std(out))

### 1a · Kay vim-1 loader (`V1–V4`)

In [ ]:
KAY_URL, KAY_PATH = "https://osf.io/ymnjv/download", "kay_images.npz"

def load_kay():
    if not os.path.exists(KAY_PATH):
        print("downloading kay_images.npz ..."); urllib.request.urlretrieve(KAY_URL, KAY_PATH)
    with np.load(KAY_PATH, allow_pickle=True) as d:
        dat = {k: d[k] for k in d.files}
    print("[kay] keys:", list(dat.keys()))
    K = dict(stim="stimuli", resp="responses", roi="roi", roi_names="roi_names")  # adapt if needed
    stim = np.asarray(dat[K["stim"]], np.float32)
    resp = np.asarray(dat[K["resp"]], np.float32)
    roi_codes = np.asarray(dat[K["roi"]]).astype(int)
    roi_names = [str(x) for x in np.asarray(dat[K["roi_names"]]).ravel()]
    print("[kay] roi_names:", roi_names, "| stim", stim.shape, "| resp", resp.shape)
    order = [r for r in ["V1", "V2", "V3", "V4"] if r in roi_names]
    masks = {r: (roi_codes == roi_names.index(r)) for r in order}
    masks = {r: m for r, m in masks.items() if m.sum() > 0}
    order = [r for r in order if r in masks]
    return VisualDataset("Kay", stim, resp, masks, order,
                         top_roi="V4" if "V4" in order else order[-1])

### 1b · NSD loader via the Algonauts 2023 dev kit (`V1–V4`)

NSD is access-gated. The tidy `subjXX/` layout below is the **Algonauts-packaged** data — from the
Algonauts 2023 registration / dev-kit **Google Drive** folder, **not** the `natural-scenes-dataset`
S3 bucket (that bucket is raw NSD: per-session betas, `.mgz` ROI atlases, a ~37 GB stimuli HDF5 — a
completely different, much heavier structure this loader does *not* read). Point `ALGONAUTS_ROOT` at
the extracted `subjXX` folder, or set `ALGONAUTS_DOWNLOAD=True` in the next cell to pull + unzip it in
Colab automatically. Expected layout:
```
subjXX/
  training_split/training_fmri/{lh,rh}_training_fmri.npy   # (N_images, N_vertices)
  training_split/training_images/*.png                      # sorted, aligned to fMRI rows
  roi_masks/mapping_prf-visualrois.npy                      # {0:'Unknown',1:'V1v',2:'V1d',...,7:'hV4'}
  roi_masks/{lh,rh}.prf-visualrois_challenge_space.npy      # vertex -> roi id
```
We collapse ventral/dorsal splits into **V1–V4** to match Kay: `V1={V1v,V1d}`, `V2={V2v,V2d}`,
`V3={V3v,V3d}`, `V4={hV4}`.


In [ ]:
ALGONAUTS_ROOT = ""            # subject folder with training_split/ + roi_masks/ (blank -> skip)
ALGONAUTS_MAX_IMAGES = 3000    # cap for Colab time/memory; set None for all (~9k)

# --- Optional Colab downloader for the Algonauts-PACKAGED data --------------------
# Pulls the tidy subjXX/ layout from the Algonauts dev-kit Google Drive folder.
# This is NOT the natural-scenes-dataset S3 bucket (raw NSD, different/heavier layout).
ALGONAUTS_SUBJ01_ZIP_ID = "1wPpPb7f9aRE4E95ZVaIq4F1CUNet3d6q"   # Algonauts subj01 zip (Drive file id)
ALGONAUTS_DOWNLOAD = False      # <-- SET True IN COLAB to fetch + unzip, then auto-set ALGONAUTS_ROOT

if ALGONAUTS_DOWNLOAD:
    import sys, subprocess, zipfile
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    if not (glob.glob("algonauts_dl/**/training_split", recursive=True) or
            glob.glob("algonauts_dl/**/test_split", recursive=True)):
        gdown.download(id=ALGONAUTS_SUBJ01_ZIP_ID, output="subj01.zip", quiet=False)  # handles large-file token
        if os.path.exists("subj01.zip"):
            print("downloaded subj01.zip: %.0f MB — unzipping..." % (os.path.getsize("subj01.zip") / 1e6))
            with zipfile.ZipFile("subj01.zip") as zf: zf.extractall("algonauts_dl")
        else:
            print("!! DOWNLOAD FAILED. Check: (a) file id is right, "
                  "(b) the Drive file is shared 'Anyone with the link'.")
    # auto-set root to the folder holding training_split (preferred) or test_split
    hits = (glob.glob("algonauts_dl/**/training_split", recursive=True) or
            glob.glob("algonauts_dl/**/test_split", recursive=True) or
            glob.glob("**/training_split", recursive=True) or
            glob.glob("**/test_split", recursive=True))
    if hits:
        ALGONAUTS_ROOT = os.path.dirname(hits[0])
        print("[nsd] ALGONAUTS_ROOT ->", ALGONAUTS_ROOT,
              "| roi_masks present:", bool(glob.glob(os.path.join(ALGONAUTS_ROOT, "**", "mapping_prf-visualrois.npy"), recursive=True)))
    else:
        print("[nsd] no *_split/ found after unzip — here is what was pulled:")
        for p in sorted(glob.glob("algonauts_dl/**", recursive=True))[:60]: print("   ", p)

def _one(root, name):                        # first recursive match for a file/dir name
    hits = glob.glob(os.path.join(root, "**", name), recursive=True)
    return hits[0] if hits else None

def _one_up(root, name, up=2):               # search root, then parent dirs (roi_masks is a SIBLING of *_split)
    r = root
    for _ in range(up + 1):
        h = glob.glob(os.path.join(r, "**", name), recursive=True)
        if h:
            return h[0]
        r = os.path.dirname(r)
    return None

def load_algonauts():
    if not ALGONAUTS_ROOT or not os.path.isdir(ALGONAUTS_ROOT):
        print("[nsd] ALGONAUTS_ROOT not set/found -> skipping (Kay-only run).")
        return None
    root = ALGONAUTS_ROOT
    # Prefer the training split (~9k images); fall back to test (~159) if that's all present.
    split = ("training" if _one(root, "lh_training_fmri.npy")
             else "test" if _one(root, "lh_test_fmri.npy") else None)
    assert split, "no lh_{training,test}_fmri.npy found under ALGONAUTS_ROOT (check the unzip)."
    if split == "test":
        print("[nsd] WARNING: only the TEST split found (~159 imgs). For the scaling arm you want "
              "the TRAINING split (~9k). NSD will sit at the LOW end of the x-axis until you add it.")
    lh = np.load(_one(root, f"lh_{split}_fmri.npy")); rh = np.load(_one(root, f"rh_{split}_fmri.npy"))
    resp = np.concatenate([lh, rh], axis=1).astype(np.float32)
    print(f"[nsd] using {split}_split | lh {lh.shape} rh {rh.shape} -> resp {resp.shape}")

    # ROI masks ship SEPARATELY from the *_split zips; find them anywhere under root.
    mapf = _one_up(root, "mapping_prf-visualrois.npy")
    assert mapf, ("mapping_prf-visualrois.npy NOT found near ALGONAUTS_ROOT.\n"
                  "roi_masks/ is a SIBLING of training_split/ inside subjXX/ -> point ALGONAUTS_ROOT "
                  "at the subjXX folder (which holds both training_split/ and roi_masks/), and re-run.")
    roi_map = np.load(mapf, allow_pickle=True).item()
    print("[nsd] prf-visualrois mapping:", roi_map)
    lh_ch = np.load(_one_up(root, "lh.prf-visualrois_challenge_space.npy"))
    rh_ch = np.load(_one_up(root, "rh.prf-visualrois_challenge_space.npy"))
    name2id = {v: k for k, v in roi_map.items()}
    groups = {"V1": ["V1v", "V1d"], "V2": ["V2v", "V2d"], "V3": ["V3v", "V3d"], "V4": ["hV4"]}
    masks, order = {}, []
    for roi, subs in groups.items():
        ids = [name2id[s] for s in subs if s in name2id]
        if not ids:
            continue
        lh_m = np.zeros(lh.shape[1], bool); rh_m = np.zeros(rh.shape[1], bool)
        for i in ids:
            lh_m |= (lh_ch == i); rh_m |= (rh_ch == i)
        m = np.concatenate([lh_m, rh_m])
        if m.sum() > 0:
            masks[roi] = m; order.append(roi)
    print("[nsd] ROI voxel counts:", {r: int(masks[r].sum()) for r in order})

    idir = _one(root, f"{split}_images")
    files = sorted(glob.glob(os.path.join(idir, "*.png")))
    if ALGONAUTS_MAX_IMAGES:
        files = files[:ALGONAUTS_MAX_IMAGES]
    resp = resp[:len(files)]
    stim = np.stack([np.array(Image.open(f).convert("RGB").resize((128, 128)))
                     for f in files]).astype(np.float32)
    print(f"[nsd] {split} stim {stim.shape} resp {resp.shape}")
    return VisualDataset(f"NSD-{split}", stim, resp, masks, order,
                         top_roi="V4" if "V4" in order else order[-1])

### 1c · Load datasets, inspect, compute noise ceilings

In [ ]:
kay = load_kay()
nsd = load_algonauts()           # None unless ALGONAUTS_ROOT is configured
DATASETS = [d for d in (kay, nsd) if d is not None]
for d in DATASETS: print(d)

# preview stimuli
fig, axes = plt.subplots(len(DATASETS), 6, figsize=(12, 2.2 * len(DATASETS)), squeeze=False)
for row, d in enumerate(DATASETS):
    for j in range(6):
        im = d.stim[j]; cmap = "gray" if im.ndim == 2 else None
        axes[row][j].imshow(im.astype(np.uint8) if im.ndim == 3 else im, cmap=cmap)
        axes[row][j].axis("off")
    axes[row][0].set_ylabel(d.name)
plt.suptitle("Stimuli per dataset"); plt.show()

# Full neural RDMs only for the BASELINE dataset (the RSA grid is the only consumer);
# building N×N RDMs for every NSD ROI would be a large, unused allocation.
BASE = DATASETS[0]
neural_rdm = {BASE.name: {r: rdm_from_responses(BASE.resp[:, BASE.roi_masks[r]]) for r in BASE.roi_order}}
# Split-half ceilings for every dataset (cheap: noise_ceiling subsamples stimuli internally).
ceiling = {}
for d in DATASETS:
    ceiling[d.name] = {r: noise_ceiling(d.resp[:, d.roi_masks[r]]) for r in d.roi_order}
    print(f"\n[{d.name}] split-half ceilings (<=1200 stim sample):")
    for r in d.roi_order: print(f"   {r:4s} {ceiling[d.name][r][0]:.3f} ± {ceiling[d.name][r][1]:.3f}")

## 2 · Backbones — frozen AlexNet & CORnet-Z

Both ImageNet-pretrained and **frozen**; forward hooks grab intermediate activations. CORnet-Z's four stages are *named* `V1/V2/V4/IT` — that naming is the entire biological inductive bias.


In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def to_model_input(imgs):
    # accept (N,H,W) gray or (N,H,W,3) color -> (N,3,224,224) ImageNet-normalized
    x = torch.as_tensor(np.asarray(imgs), dtype=torch.float32)
    if x.dim() == 3:
        x = x.unsqueeze(1)
    elif x.dim() == 4 and x.shape[-1] == 3:
        x = x.permute(0, 3, 1, 2)
    if x.shape[1] == 1:
        x = x.repeat(1, 3, 1, 1)
    if x.max() > 1.5:
        x = x / 255.0
    x = F.interpolate(x, size=224, mode="bilinear", align_corners=False)
    return (x - IMAGENET_MEAN) / IMAGENET_STD

In [ ]:
class FeatureExtractor:
    # register hooks on named submodules; returns {layer: activation} per forward
    def __init__(self, model, layers):
        self.model = model.eval().to(DEVICE)
        for p in self.model.parameters(): p.requires_grad_(False)
        self.layers = layers; self._buf = {}
        self._handles = [m.register_forward_hook(self._mk(n)) for n, m in layers.items()]
    def _mk(self, name):
        def hook(_m, _i, out): self._buf[name] = out
        return hook
    @torch.no_grad()
    def activations(self, x):
        self._buf = {}; self.model(x)
        return {k: v.detach() for k, v in self._buf.items()}
    def remove(self):                      # detach hooks so stale closures don't keep firing / retaining tensors
        for h in self._handles: h.remove()
        self._handles = []
    def __enter__(self): return self
    def __exit__(self, *a): self.remove()

def flat_features(model, layers, imgs, batch=64):
    # {layer: (N,D)} global-avg-pooled if 4D; hooks auto-removed on exit
    outs = {k: [] for k in layers}
    with FeatureExtractor(model, layers) as ex:
        for i in range(0, len(imgs), batch):
            acts = ex.activations(to_model_input(imgs[i:i + batch]).to(DEVICE))
            for k, v in acts.items():
                if v.dim() == 4: v = v.mean(dim=(2, 3))
                outs[k].append(v.flatten(1).cpu())
    return {k: torch.cat(v).numpy() for k, v in outs.items()}

In [ ]:
alexnet = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.IMAGENET1K_V1)
alex_layers = {"conv1": alexnet.features[0], "conv2": alexnet.features[3],
               "conv3": alexnet.features[6], "conv4": alexnet.features[8],
               "conv5": alexnet.features[10], "fc6": alexnet.classifier[2],
               "fc7": alexnet.classifier[5]}

import cornet
cornet_z = cornet.cornet_z(pretrained=True, map_location=DEVICE)
cz = getattr(cornet_z, "module", cornet_z)
cornet_layers = {"V1": cz.V1, "V2": cz.V2, "V4": cz.V4, "IT": cz.IT}

# 3rd item = spatial (C,H,W) map the attention head will pool over
BACKBONES = {"AlexNet": (alexnet, alex_layers, alexnet.features[10]),
             "CORnet-Z": (cornet_z, cornet_layers, cz.V4)}
print({k: list(v[1]) for k, v in BACKBONES.items()})

## 3 · RSA baseline (no training) — does the hierarchy already line up?

Per backbone layer, build a model RDM and correlate with each ROI's neural RDM → a **layer × ROI grid**. Expect a rough diagonal for AlexNet (early→V1, late→V4; the Güçlü & van Gerven 2015 result) and a *cleaner, more monotonic* diagonal for CORnet-Z's named stages.


In [ ]:
def alignment_grid(feats, d):
    layers = list(feats); G = np.zeros((len(layers), len(d.roi_order)))
    for i, lyr in enumerate(layers):
        mr = rdm_from_responses(feats[lyr])
        for j, r in enumerate(d.roi_order):
            G[i, j] = rsa(mr, neural_rdm[d.name][r])
    return layers, G

grids = {}                # BASE defined in cell 1c (= DATASETS[0], Kay by default)
for bname, (model, layers, _) in BACKBONES.items():
    feats = flat_features(model, layers, BASE.stim)
    grids[bname] = alignment_grid(feats, BASE)
print("RSA grids computed for", list(grids), "on", BASE.name)

In [ ]:
def plot_grid(ax, layers, G, rois, title):
    im = ax.imshow(G, cmap="magma", aspect="auto", vmin=0, vmax=max(0.01, G.max()))
    ax.set_xticks(range(len(rois))); ax.set_xticklabels(rois)
    ax.set_yticks(range(len(layers))); ax.set_yticklabels(layers)
    ax.set_xlabel("neural ROI"); ax.set_ylabel("model layer"); ax.set_title(title)
    for i in range(G.shape[0]):
        for j in range(G.shape[1]):
            ax.text(j, i, f"{G[i,j]:.2f}", ha="center", va="center",
                    color="w" if G[i, j] < G.max() * 0.6 else "k", fontsize=8)
    return im

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, (bname, (layers, G)) in zip(axes, grids.items()):
    im = plot_grid(ax, layers, G, BASE.roi_order, f"{bname} · {BASE.name}")
fig.colorbar(im, ax=axes, shrink=0.8, label="Spearman RSA")
plt.show()
print("ceilings:", {r: round(ceiling[BASE.name][r][0], 3) for r in BASE.roi_order})

**Slide takeaway.** A tighter, more monotonic diagonal for CORnet-Z (named stages peaking on the matching ROI) is evidence its geometry is *already* organized like cortex — before any training.

## 4 · The intervention — a trainable attention head (the DL piece)

In [ ]:
@torch.no_grad()
def spatial_map(model, map_module, imgs, batch=64):
    outs = []
    with FeatureExtractor(model, {"map": map_module}) as fx:   # hooks removed on exit
        for i in range(0, len(imgs), batch):
            outs.append(fx.activations(to_model_input(imgs[i:i + batch]).to(DEVICE))["map"].cpu())
    return torch.cat(outs)

class AttentionHead(nn.Module):
    # attention-pool a (C,H,W) map to an embedding via a learned query
    def __init__(self, in_ch, embed_dim=256, n_heads=4):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, 1)
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.attn = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, fmap):
        x = self.proj(fmap); b, dd, h, w = x.shape
        tok = x.flatten(2).transpose(1, 2)
        q = self.query.expand(b, -1, -1)
        pooled, _ = self.attn(q, tok, tok)
        return self.norm(pooled.squeeze(1))

In [ ]:
def torch_rdm(x):
    x = x - x.mean(dim=1, keepdim=True); x = F.normalize(x, dim=1)
    return 1.0 - x @ x.t()
def torch_upper(m):
    iu = torch.triu_indices(m.size(0), m.size(0), offset=1, device=m.device)
    return m[iu[0], iu[1]]
def rsa_loss(emb, target_rdm):
    mv, tv = torch_upper(torch_rdm(emb)), torch_upper(target_rdm)
    mv = mv - mv.mean(); tv = tv - tv.mean()
    return 1.0 - (mv @ tv) / (mv.norm() * tv.norm() + 1e-8)

## 5 · Train both arms (generalized over dataset × backbone)

Identical head/optimizer/schedule; only the frozen backbone differs. Trains the head so its geometry matches the dataset's **top ventral ROI** (`V4`), logging train vs held-out RSA.


In [ ]:
EMBED_DIM, EPOCHS, BATCH, LR = 256, 40, 96, 1e-3
EVAL_N = 600      # fixed sample for train/held RSA: bounds the N×N RDM and keeps the gap comparable

def split_idx(n, frac_hold=0.2):
    rng = np.random.default_rng(SEED); p = rng.permutation(n); k = int(n * frac_hold)
    return p[k:], p[:k]

def train_arm(d, bname, epochs=EPOCHS, subset=None, verbose=True):
    model, _, map_module = BACKBONES[bname]
    maps = spatial_map(model, map_module, d.stim)
    resp_roi = d.resp[:, d.roi_masks[d.top_roi]]
    tr_idx, hold_idx = split_idx(len(d.stim))
    if subset: tr_idx = tr_idx[:subset]
    tr_eval = tr_idx[:min(len(tr_idx), EVAL_N)]        # sample of the TRAIN split for train RSA
    hold_eval = hold_idx[:min(len(hold_idx), EVAL_N)]  # sample of the HELD-OUT split
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)   # identical head init across paired arms/subsets
    head = AttentionHead(maps.shape[1], EMBED_DIM).to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=LR, weight_decay=1e-4)
    hist = {"epoch": [], "train": [], "hold": []}
    def eval_rsa(idx):
        head.eval()
        with torch.no_grad():
            emb = head(maps[idx].to(DEVICE)).cpu().numpy()
        return rsa(rdm_from_responses(emb), rdm_from_responses(resp_roi[idx]))
    for ep in range(epochs):
        head.train()
        for i in range(0, len(tr_idx), BATCH):
            b = tr_idx[i:i + BATCH]
            if len(b) < 8: continue
            tgt = torch.as_tensor(rdm_from_responses(resp_roi[b]), dtype=torch.float32, device=DEVICE)
            loss = rsa_loss(head(maps[b].to(DEVICE)), tgt)
            opt.zero_grad(); loss.backward(); opt.step()
        if ep % 5 == 0 or ep == epochs - 1:
            hist["epoch"].append(ep)
            hist["train"].append(eval_rsa(tr_eval)); hist["hold"].append(eval_rsa(hold_eval))
            if verbose: print(f"[{d.name}/{bname}] ep{ep:2d} train {hist['train'][-1]:+.3f} held {hist['hold'][-1]:+.3f}")
    return hist

hists = {b: train_arm(BASE, b) for b in BACKBONES}

## 6 · Results on the baseline dataset — alignment & the overfitting gap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = {"AlexNet": "#d1495b", "CORnet-Z": "#2e86ab"}
ax = axes[0]
for b, h in hists.items():
    ax.plot(h["epoch"], h["train"], "--", color=colors[b], alpha=0.6)
    ax.plot(h["epoch"], h["hold"], "-", color=colors[b], label=b)
ax.axhline(ceiling[BASE.name][BASE.top_roi][0], ls=":", color="gray", label="noise ceiling")
ax.set_xlabel("epoch"); ax.set_ylabel(f"RSA to {BASE.top_roi}")
ax.set_title(f"{BASE.name}: train (dashed) vs held-out (solid)"); ax.legend(fontsize=8)

ax = axes[1]; names = list(hists); x = np.arange(len(names))
hold_final = [hists[b]["hold"][-1] for b in names]
gap = [hists[b]["train"][-1] - hists[b]["hold"][-1] for b in names]
ax.bar(x - 0.2, hold_final, 0.4, label="held-out RSA", color=[colors[b] for b in names])
ax.bar(x + 0.2, gap, 0.4, label="train−held gap", color="#bbb")
ax.axhline(ceiling[BASE.name][BASE.top_roi][0], ls=":", color="gray")
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_title("Final"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
for b in names:
    print(f"{b}: held-out {hists[b]['hold'][-1]:+.3f} | gap {gap[names.index(b)]:+.3f} (smaller=better)")

## 7 · The main figure — does the CORnet advantage shrink with scale?

The identical head is trained at increasing training-set sizes, **spanning both datasets on one x-axis** (# images, log scale). Prediction: CORnet-Z leads AlexNet at Kay-scale and the gap **narrows** toward NSD-scale. A narrowing gap = evidence the biological prior behaves like a *regularizer*; a constant gap = it's just a better model.


In [ ]:
def scaling_points(d, epochs=25):
    n_tr = int(len(d.stim) * 0.8)
    sizes = sorted(set(max(60, int(f * n_tr)) for f in [0.1, 0.25, 0.5, 1.0]))
    rows = []
    for s in sizes:
        pt = {"dataset": d.name, "n": s}
        for b in BACKBONES:
            pt[b] = train_arm(d, b, epochs=epochs, subset=s, verbose=False)["hold"][-1]
        rows.append(pt); print(f"[{d.name}] n={s}: " + " ".join(f"{b}={pt[b]:+.3f}" for b in BACKBONES))
    return rows

all_pts = []
for d in DATASETS:
    all_pts += scaling_points(d)

plt.figure(figsize=(7, 4.6))
dnames = sorted(set(p["dataset"] for p in all_pts))
for b, c in colors.items():
    for dname in dnames:                       # connect ONLY within a dataset (absolute RSA differs by subject/scanner)
        pts = sorted([p for p in all_pts if p["dataset"] == dname], key=lambda p: p["n"])
        xs = np.array([p["n"] for p in pts]); ys = np.array([p[b] for p in pts])
        plt.plot(xs, ys, "-", color=c, alpha=0.5)
        plt.scatter(xs, ys, color=c, s=60, marker=("o" if dname == "Kay" else "s"),
                    edgecolor="k", linewidth=0.5, zorder=3)
plt.xscale("log"); plt.xlabel("# training images (log)"); plt.ylabel("held-out RSA to V4")
plt.title("Regularization across scale — ○ Kay   □ NSD")
handles = [plt.Line2D([], [], color=c, marker="s", label=b) for b, c in colors.items()]
plt.legend(handles=handles, fontsize=9); plt.tight_layout(); plt.show()
if len(DATASETS) == 1:
    print("NOTE: NSD not configured — showing the within-Kay curve only. "
          "Set ALGONAUTS_ROOT in cell 1b to extend the x-axis to ~9k images.")

## 8 · Interpretation, limitations, next steps

**Reading the main figure (both outcomes are valid results):**
- **Hypothesis supported** — CORnet-Z leads at Kay-scale *and the lead shrinks* by NSD-scale: the biological prior is acting as a data-efficient regularizer.
- **Not supported** — the gap is flat (a constant model-quality difference) or absent: then "biological hardware" buys nothing beyond ordinary architecture choices. A clean null is a good NMA result — say so.

**Why we did NOT augment images:** BOLD responses are stimulus-locked; augmenting the image breaks the (image, response) pairing and corrupts the RSA target. Adding a second *real* dataset is the sound way to grow N — and it doubles as the scaling axis the hypothesis needs.

**Limitations for the honesty slide:**
- Kay and NSD are different subjects/scanners, so *absolute* RSA differs — we read the *within-dataset scaling trend*, never merged voxels. Both expose retinotopic V1–V4, so the comparison is V4→V4.
- Split-half noise ceiling is a proxy, estimated on a capped stimulus sample (≤1200) for tractability. NSD provides up to 3 repeats per image → a proper test–retest ceiling is a good upgrade.
- NSD stimuli are natural color scenes vs Kay's grayscale; preprocessing normalizes both but input statistics differ.
- One trained target ROI at a time; a single embedding can't match early and late geometry at once (interesting tension, not a bug).
- We cap NSD at `ALGONAUTS_MAX_IMAGES` for Colab; raise it (or add subjects) for the full scaling range.

**Highest-value extensions if time remains:**
1. Add **CORnet-S** (recurrent, higher Brain-Score) as a third arm — is more brain-like = more regularized?
2. Proper test–retest noise ceiling from NSD repeats.
3. Per-ROI alignment profiles (loop `top_roi` over the full `roi_order`).
4. Head ablation: attention-pool vs mean-pool vs linear — isolate the attention's contribution.

### References
- Kay, Naselaris, Prenger & Gallant (2008). *Identifying natural images from human brain activity.* **Nature** 452:352.
- Allen, St-Yves, Wu, ... Kay & Naselaris (2022). *A massive 7T fMRI dataset (NSD).* **Nature Neuroscience** 25:116.
- Gifford, Lahner, Saba-Sadiya et al. (2023). *The Algonauts Project 2023 Challenge.* arXiv:2301.03198.
- Güçlü & van Gerven (2015). *A gradient in the complexity of neural representations across the ventral stream.* **J. Neurosci.** 35:10005.
- Kriegeskorte, Mur & Bandettini (2008) / Nili et al. (2014) — RSA methodology & noise ceilings.
- Kubilius, Schrimpf et al. (2019). *Brain-like object recognition with high-performing shallow recurrent ANNs (CORnet).* **NeurIPS**.
